File to play around with data visualization to plan out the model. Currently I am thinking:
The problem can be split into two parts:
1. The model needs to understand the combinations of ingredients and their quantities
2. The model needs to learn how to cook the ingredients

A lot of nuances:
The combination of the ingredients depends on its state (raw, cooked, etc.)
The combination of the ingredients depends on the cooking method (frying, baking, etc.)
Cooking the ingredients is dependent on the cooking combiantion.

I thought of splitting the problem into two parts to have two types of models to tackle the problem. But will need more research and EDA to figure out the best way to tackle this problem.

In [25]:
# import the libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import csv
import os
from IPython.display import display, Markdown

In [ ]:
def flatten_recipe_json(json_filepath, csv_filepath):
    """
    Reads a Scrapy recipe JSON/JSONL export, flattens the nested structures, 
    and outputs a clean CSV optimized for Excel/Google Sheets.
    """
    if not os.path.exists(json_filepath):
        display(Markdown(f"Error. The file `{json_filepath}` does not exist. Check your path."))
        return

    # Load JSON data
    with open(json_filepath, 'r', encoding='utf-8') as f:
        try:
            data = json.load(f)
        except json.JSONDecodeError:
            # Fallback wrapper for JSON Lines (.jsonl format)
            f.seek(0)
            data = [json.loads(line) for line in f if line.strip()]

    if not data:
        display(Markdown("Warning. The JSON file is empty."))
        return

    if isinstance(data, dict):
        data = [data]

    flattened_rows = []
    for recipe in data:
        # 1. Flatten nested Ingredients array
        flat_ingredients = []
        for group in recipe.get('ingredient_groups', []):
            group_name = group.get('group_name') or "Main"
            flat_ingredients.append(f"--- {group_name.upper()} ---")
            for ing in group.get('ingredients', []):
                qty = f"{ing.get('amount', '')} {ing.get('unit', '')}".strip() or "-"
                notes = f" ({ing.get('notes')})" if ing.get('notes') else ""
                flat_ingredients.append(f"{qty} | {ing.get('name', '')}{notes}")
        
        # 2. Flatten nested Instructions array
        flat_instructions = []
        for group in recipe.get('instruction_groups', []):
            group_name = group.get('group_name') or "General"
            flat_instructions.append(f"--- {group_name.upper()} ---")
            for i, step in enumerate(group.get('instructions', []), start=1):
                flat_instructions.append(f"Step {i:02d}: {step}")

        # Map to unified structure using clean spreadsheet row mapping rules
        row = {
            'Recipe Name': recipe.get('recipe_name', ''),
            'Servings': recipe.get('servings', ''),
            'Ingredients': "\n".join(flat_ingredients),
            'Instructions': "\n".join(flat_instructions),
            'Notes': "\n".join(recipe.get('notes', []))
        }
        flattened_rows.append(row)

    # Output to disk
    headers = ['Recipe Name', 'Servings', 'Ingredients', 'Instructions', 'Notes']
    with open(csv_filepath, 'w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        writer.writeheader()
        writer.writerows(flattened_rows)

    # Notebook-optimized rich text confirmation
    display(Markdown(f"Conversion Successful!\n* **Input:** `{json_filepath}`\n* **Output:** `{csv_filepath}`\n* **Total Records Processed:** `{len(flattened_rows)}` recipe rows."))

In [26]:
# Configuration variables
INPUT_JSON = "/home/raj/Code/Project_Ramsay/recipe_crawler/recipes.json"   # Switch extension to .jsonl if using JSON Lines format
OUTPUT_CSV = "recipes.csv"

# Run the execution processing engine
flatten_recipe_json(INPUT_JSON, OUTPUT_CSV)

### 🎉 Conversion Successful!
* **Input:** `/home/raj/Code/Project_Ramsay/recipe_crawler/recipes.json`
* **Output:** `recipes.csv`
* **Total Records Processed:** `1482` recipe rows.

In [29]:
df = pd.read_csv("recipes.csv")
df.head()

,Recipe Name,Servings,Ingredients,Instructions,Notes
0,Nagi’s Salad Sandwich,1.0,"--- MAIN ---\n1 | soft bread roll (, wholemeal...",--- GENERAL ---\nStep 01: Dress carrot first –...,Best made and eaten immediately. This is not a...
1,Galettes Bretonnes (Brittany Butter Biscuits),24.0,--- MAIN ---\n180g / 12 tbsp | salted butter (...,--- ABBREVIATED RECIPE: ---\nStep 01: Cream bu...,1. Butter – These biscuits are traditionally m...
2,Coconut Chicken Curry – quick and easy,5.0,--- MAIN ---\n2 tbsp | unrefined coconut oil (...,"--- ABBREVIATED ---\nStep 01: Melt oil, sauté ...",1. Coconut oil – Use unrefined (virgin) coconu...
3,Tuna Carpaccio,2.0,"--- TUNA ---\n120g / 4.2oz g | tuna (, skinles...",--- ABBREVIATED RECIPE: ---\nStep 01: Mix onio...,"1. Tuna – This dish is served raw, so using a ..."
4,Coquillettes au Jambon – French Macaroni and C...,5.0,--- MAIN ---\n500g / 1 lb | elbow pasta / maca...,--- ABBREVIATED RECIPE: ---\nStep 01: Cook pas...,1. Ham options – Any good quality ham works he...


In [30]:
print(len(df))

1482
